# 10 — Advanced AML Analytics Dashboard

Comprehensive analytics with **three specialized tabs**:

| Tab | Purpose |
|-----|----------|
| **AML Scores** | 5-signal composite scoring (SAR, Laundering, Network, Geo, Volume) |
| **Cases** | Investigation cases with typology detection (Fan-Out, Fan-In, Circular, Hub, SAR Proximity) |
| **Risk Ranking** | Tiered risk queue (T1-T4) with percentile ranking |

Run after notebooks 01–05 to analyze model predictions with production-grade scoring.

In [ ]:
import os
import json
import math
import warnings
from collections import defaultdict
from pathlib import Path
from typing import Dict, Set, List, Any, Tuple
from uuid import uuid4

import numpy as np
import pandas as pd
import torch
import networkx as nx

import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

import dash
from dash import dcc, html, dash_table, Input, Output, State
import dash_bootstrap_components as dbc

from gan_anomaly import Generator, Encoder, anomaly_score as compute_anomaly_score

warnings.filterwarnings('ignore')
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ── Pipeline integration ──────────────────────────────────────────
_RUN_DIR = os.environ.get("AML_RUN_DIR", "")
DATA_DIR = os.path.join(_RUN_DIR, "data") if _RUN_DIR else "data"
EMBEDDINGS_DIR = os.path.join(_RUN_DIR, "embeddings") if _RUN_DIR else "embeddings"
MODELS_DIR = os.path.join(_RUN_DIR, "models") if _RUN_DIR else "models"
QUEUES_DIR = os.path.join(_RUN_DIR, "queues") if _RUN_DIR else "results/queues"
CASES_DIR = os.path.join(_RUN_DIR, "cases") if _RUN_DIR else "results/cases"

os.makedirs(QUEUES_DIR, exist_ok=True)
os.makedirs(CASES_DIR, exist_ok=True)

print(f"Device: {device}")

---
## Load Data & Models

In [ ]:
# Load processed data
transactions = pd.read_parquet(os.path.join(DATA_DIR, "transactions_processed.parquet"))
node_features = pd.read_parquet(os.path.join(DATA_DIR, "node_features.parquet"))
alert_nodes = pd.read_parquet(os.path.join(DATA_DIR, "alert_nodes.parquet"))
edges = pd.read_parquet(os.path.join(DATA_DIR, "edges.parquet"))

# Load GAN models and threshold
with open(os.path.join(MODELS_DIR, "training_meta.json")) as f:
    meta = json.load(f)

G_model = Generator(meta["latent_dim"], meta["input_dim"], meta["g_hidden"], meta["n_layers"], meta["activation"]).to(device)
E_model = Encoder(meta["input_dim"], meta["latent_dim"], meta["e_hidden"], meta["n_layers"], meta["activation"]).to(device)
G_model.load_state_dict(torch.load(os.path.join(MODELS_DIR, "generator.pt"), map_location=device, weights_only=True))
E_model.load_state_dict(torch.load(os.path.join(MODELS_DIR, "encoder.pt"), map_location=device, weights_only=True))

# Load embeddings and compute anomaly scores
embeddings = np.load(os.path.join(EMBEDDINGS_DIR, "node_embeddings.npy"))
node_ids = np.load(os.path.join(EMBEDDINGS_DIR, "node_ids.npy"), allow_pickle=True)

X_train = np.load(os.path.join(MODELS_DIR, "X_train.npy"))
train_scores = compute_anomaly_score(
    torch.tensor(X_train, dtype=torch.float32).to(device), E_model, G_model
).cpu().numpy()
threshold = float(np.percentile(train_scores, 99))

all_scores = compute_anomaly_score(
    torch.tensor(embeddings, dtype=torch.float32).to(device), E_model, G_model
).cpu().numpy()

print(f"Loaded {len(transactions):,} transactions, {len(node_features):,} nodes")
print(f"Anomaly threshold (P99): {threshold:.6f}")

---
## 1. Risk Ranking Logic

**Formula:** `risk_score = 2.0 * is_sar + 0.5 * log1p(degree)`

**Tiers:**
- T1: top 0.5% (Critical)
- T2: 0.5–2% (High)
- T3: 2–5% (Medium)
- T4: rest (Low)

In [3]:
TIER_THRESHOLDS = {"T1": 0.5, "T2": 2.0, "T3": 5.0}

def assign_tier(percentile):
    if percentile <= TIER_THRESHOLDS["T1"]:
        return "T1"
    elif percentile <= TIER_THRESHOLDS["T2"]:
        return "T2"
    elif percentile <= TIER_THRESHOLDS["T3"]:
        return "T3"
    return "T4"

def build_risk_queue(node_df, edges_df):
    """Build tiered risk queue from node features."""
    # Compute undirected degree
    src_counts = edges_df["source"].value_counts()
    dst_counts = edges_df["target"].value_counts()
    degree = src_counts.add(dst_counts, fill_value=0).astype(int)
    
    queue = node_df[["id", "type", "is_sar"]].copy()
    queue["entity_id"] = queue["id"].astype(str)
    queue["entity_type"] = queue["type"].map({0: "Organization", 1: "Individual"}).fillna("Unknown")
    queue["degree"] = queue["id"].map(degree).fillna(0).astype(int)
    
    # Risk score formula
    queue["risk_score"] = 2.0 * queue["is_sar"] + 0.5 * queue["degree"].apply(lambda d: math.log1p(d))
    
    # Rank and percentile
    queue = queue.sort_values("risk_score", ascending=False).reset_index(drop=True)
    total = len(queue)
    queue["rank"] = range(1, total + 1)
    queue["percentile"] = ((queue["rank"] - 1) / total) * 100
    queue["tier"] = queue["percentile"].apply(assign_tier)
    
    # Reasons
    def build_reasons(row):
        reasons = []
        if row["is_sar"] == 1:
            reasons.append("SAR_LABEL")
        if row["degree"] > 0:
            reasons.append(f"CONNECTIVITY(deg={row['degree']})")
        return "; ".join(reasons)
    
    queue["reasons"] = queue.apply(build_reasons, axis=1)
    
    return queue[["entity_id", "entity_type", "is_sar", "degree", "risk_score", "rank", "percentile", "tier", "reasons"]]

risk_queue = build_risk_queue(node_features, edges)
print(f"Risk queue built: {len(risk_queue):,} entities")
print(f"Tier distribution: {risk_queue['tier'].value_counts().to_dict()}")

Risk queue built: 7,500 entities
Tier distribution: {'T4': 7124, 'T3': 225, 'T2': 113, 'T1': 38}


---
## 2. AML Scoring Logic

**5-Signal Composite Score (0–100):**

| Signal | Weight | Calculation |
|--------|--------|-------------|
| SAR | 3.0 | Binary (is_sar) |
| Laundering | 2.5 | laundering_txn_count / txn_count |
| Network | 1.5 | 0.5*log(degree)/log(max) + 0.5*fanout_ratio |
| Geo | 1.0 | 0.6*cross_border_rate + 0.4*jurisdiction_diversity |
| Volume | 0.5 | total_amt / p99_amt |

In [4]:
WEIGHTS = {"W_SAR": 3.0, "W_LAUN": 2.5, "W_NET": 1.5, "W_GEO": 1.0, "W_VOL": 0.5}
RISK_BANDS = [(80, 100, "Critical"), (60, 80, "High"), (30, 60, "Medium"), (0, 30, "Low")]

def assign_band(score):
    for lo, hi, label in RISK_BANDS:
        if lo <= score <= hi:
            return label
    return "Low"

def compute_aml_scores(queue_df, transactions_df, edges_df):
    """Compute 5-signal AML scores for all entities."""
    scored = queue_df.copy()
    
    # Transaction aggregates per entity
    out_agg = transactions_df.groupby("source").agg(
        out_txn_count=("tran_id", "count"),
        out_total_amt=("base_amt", "sum")
    )
    in_agg = transactions_df.groupby("target").agg(
        in_txn_count=("tran_id", "count"),
        in_total_amt=("base_amt", "sum")
    )
    
    scored = scored.merge(out_agg, left_on="entity_id", right_index=True, how="left")
    scored = scored.merge(in_agg, left_on="entity_id", right_index=True, how="left")
    scored = scored.fillna(0)
    
    scored["txn_count"] = scored["out_txn_count"] + scored["in_txn_count"]
    scored["total_amt"] = scored["out_total_amt"] + scored["in_total_amt"]
    
    # Fan-out ratio
    out_degree = edges_df["source"].value_counts()
    in_degree = edges_df["target"].value_counts()
    total_degree = out_degree.add(in_degree, fill_value=0)
    fanout_ratio = out_degree.divide(total_degree.clip(lower=1)).fillna(0)
    scored["fanout_ratio"] = scored["entity_id"].map(fanout_ratio).fillna(0.0)
    
    # ── Signal 1: SAR (binary) ──
    scored["sar_signal"] = scored["is_sar"].astype(float).clip(0, 1)
    
    # ── Signal 2: Laundering (we don't have this data, default to 0) ──
    scored["laundering_signal"] = 0.0
    
    # ── Signal 3: Network ──
    max_degree = max(scored["degree"].max(), 1)
    scored["network_signal"] = (
        0.5 * scored["degree"].apply(lambda d: math.log1p(d)) / math.log1p(max_degree)
        + 0.5 * scored["fanout_ratio"]
    ).clip(0, 1)
    
    # ── Signal 4: Geo (we don't have cross-border data, default to 0) ──
    scored["geo_signal"] = 0.0
    
    # ── Signal 5: Volume ──
    p99_amt = scored["total_amt"].quantile(0.99) if scored["total_amt"].max() > 0 else 1.0
    p99_amt = max(p99_amt, 1.0)
    scored["volume_signal"] = (scored["total_amt"] / p99_amt).clip(0, 1)
    
    # ── Composite Score ──
    # Active weights: SAR + Network + Volume (laundering/geo unavailable)
    active_weight = WEIGHTS["W_SAR"] + WEIGHTS["W_NET"] + WEIGHTS["W_VOL"]
    
    scored["_raw_score"] = (
        WEIGHTS["W_SAR"] * scored["sar_signal"]
        + WEIGHTS["W_NET"] * scored["network_signal"]
        + WEIGHTS["W_VOL"] * scored["volume_signal"]
    )
    scored["aml_score"] = ((scored["_raw_score"] / active_weight) * 100).round(2)
    scored["risk_band"] = scored["aml_score"].apply(assign_band)
    
    # Reasons
    def build_aml_reasons(row):
        reasons = []
        if row["sar_signal"] > 0:
            reasons.append("SAR_LABEL")
        if row["network_signal"] > 0.5:
            reasons.append(f"HIGH_CONNECTIVITY(deg={int(row['degree'])})")
        if row["volume_signal"] > 0.8:
            reasons.append("HIGH_VOLUME")
        return ", ".join(reasons)
    
    scored["aml_reasons"] = scored.apply(build_aml_reasons, axis=1)
    
    return scored.sort_values("aml_score", ascending=False).reset_index(drop=True)

aml_scores = compute_aml_scores(risk_queue, transactions, edges)
print(f"AML scores computed: {len(aml_scores):,} entities")
print(f"Band distribution: {aml_scores['risk_band'].value_counts().to_dict()}")

AML scores computed: 7,500 entities
Band distribution: {'Low': 6834, 'High': 632, 'Medium': 22, 'Critical': 12}


---
## 3. Case Builder Logic

**Typologies Detected:**
- FAN_OUT: out_degree ≥ 3, out/in ratio ≥ 3.0
- FAN_IN: in_degree ≥ 3, in/out ratio ≥ 3.0
- CIRCULAR_FLOW: cycles ≤ 3 hops
- HUB_DOMINANCE: max_degree/avg_degree ≥ 5.0
- SAR_PROXIMITY: SAR entity in subgraph

In [5]:
FAN_RATIO_THRESHOLD = 3.0
HUB_DOMINANCE_THRESHOLD = 5.0
CYCLE_MAX_LENGTH = 3
SEED_TIERS = ["T1", "T2"]
MAX_PERCENTILE = 2.0
HOP_K = 2

def build_adjacency(edges_df):
    """Build undirected adjacency dict."""
    adj = defaultdict(set)
    for _, row in edges_df.iterrows():
        adj[str(row["source"])].add(str(row["target"]))
        adj[str(row["target"])].add(str(row["source"]))
    return dict(adj)

def k_hop_neighbors(seed, adj, k):
    """Get all nodes within k hops of seed."""
    visited = {seed}
    frontier = {seed}
    for _ in range(k):
        next_frontier = set()
        for node in frontier:
            for neighbor in adj.get(node, set()):
                if neighbor not in visited:
                    visited.add(neighbor)
                    next_frontier.add(neighbor)
        frontier = next_frontier
        if not frontier:
            break
    return visited

def detect_typologies(nodes, sub_edges_df, sar_ids):
    """Detect AML typologies in a subgraph."""
    typologies = []
    
    if sub_edges_df.empty:
        if nodes & sar_ids:
            typologies.append("SAR_PROXIMITY")
        return typologies
    
    # Compute directed degrees
    out_deg = sub_edges_df["source"].value_counts().to_dict()
    in_deg = sub_edges_df["target"].value_counts().to_dict()
    
    # FAN_OUT
    for n in nodes:
        od = out_deg.get(n, 0)
        ind = in_deg.get(n, 0)
        if od >= 3 and (ind == 0 or od / max(ind, 1) >= FAN_RATIO_THRESHOLD):
            typologies.append("FAN_OUT")
            break
    
    # FAN_IN
    for n in nodes:
        ind = in_deg.get(n, 0)
        od = out_deg.get(n, 0)
        if ind >= 3 and (od == 0 or ind / max(od, 1) >= FAN_RATIO_THRESHOLD):
            typologies.append("FAN_IN")
            break
    
    # CIRCULAR_FLOW (simplified cycle detection)
    adj_out = defaultdict(set)
    for _, row in sub_edges_df.iterrows():
        adj_out[str(row["source"])].add(str(row["target"]))
    
    found_cycle = False
    for start in list(nodes)[:50]:  # Limit search
        if found_cycle:
            break
        stack = [(start, [start])]
        while stack and not found_cycle:
            current, path = stack.pop()
            for neighbor in adj_out.get(current, set()):
                if neighbor == start and len(path) >= 2:
                    found_cycle = True
                    break
                if neighbor not in path and len(path) < CYCLE_MAX_LENGTH:
                    stack.append((neighbor, path + [neighbor]))
    
    if found_cycle:
        typologies.append("CIRCULAR_FLOW")
    
    # HUB_DOMINANCE
    total_deg = {n: out_deg.get(n, 0) + in_deg.get(n, 0) for n in nodes}
    if total_deg:
        max_d = max(total_deg.values())
        avg_d = sum(total_deg.values()) / len(total_deg)
        if avg_d > 0 and max_d / avg_d >= HUB_DOMINANCE_THRESHOLD:
            typologies.append("HUB_DOMINANCE")
    
    # SAR_PROXIMITY
    if nodes & sar_ids:
        typologies.append("SAR_PROXIMITY")
    
    return sorted(set(typologies))

def build_cases(queue_df, edges_df):
    """Build investigation cases from risk queue."""
    # Select seeds
    seeds_df = queue_df[queue_df["percentile"] <= MAX_PERCENTILE].copy()
    seeds_df = seeds_df[seeds_df["tier"].isin(SEED_TIERS)]
    seed_ids = set(seeds_df["entity_id"].astype(str))
    
    if not seed_ids:
        return pd.DataFrame(), {}
    
    # Build adjacency
    adj = build_adjacency(edges_df)
    sar_ids = set(queue_df[queue_df["is_sar"] == 1]["entity_id"].astype(str))
    queue_lookup = queue_df.set_index("entity_id").to_dict("index")
    
    # Build ego graphs per seed
    raw_cases = []
    for seed_id in seed_ids:
        nodes = k_hop_neighbors(seed_id, adj, HOP_K)
        raw_cases.append({"seeds": {seed_id}, "nodes": nodes})
    
    # Merge overlapping cases (simple approach)
    merged = []
    used = set()
    for i, case_i in enumerate(raw_cases):
        if i in used:
            continue
        combined_seeds = case_i["seeds"].copy()
        combined_nodes = case_i["nodes"].copy()
        for j, case_j in enumerate(raw_cases[i+1:], i+1):
            if j in used:
                continue
            overlap = len(combined_nodes & case_j["nodes"])
            if overlap > 0:
                combined_seeds |= case_j["seeds"]
                combined_nodes |= case_j["nodes"]
                used.add(j)
        merged.append({"seeds": combined_seeds, "nodes": combined_nodes})
        used.add(i)
    
    # Build case records
    tier_priority = {"T1": 0, "T2": 1, "T3": 2, "T4": 3}
    case_records = []
    
    for raw in merged:
        case_id = str(uuid4())[:8]
        nodes = raw["nodes"]
        seeds = raw["seeds"]
        
        # Subgraph edges
        sub_edges = edges_df[
            edges_df["source"].astype(str).isin(nodes) & 
            edges_df["target"].astype(str).isin(nodes)
        ]
        
        # Best tier and risk score
        best_tier = "T4"
        max_risk = 0.0
        for s in seeds:
            info = queue_lookup.get(s, {})
            s_tier = info.get("tier", "T4")
            if tier_priority.get(s_tier, 3) < tier_priority.get(best_tier, 3):
                best_tier = s_tier
            max_risk = max(max_risk, info.get("risk_score", 0.0))
        
        # Typologies
        typologies = detect_typologies(nodes, sub_edges, sar_ids)
        sar_in_case = len(nodes & sar_ids)
        
        case_records.append({
            "case_id": case_id,
            "tier": best_tier,
            "risk_score": round(max_risk, 4),
            "entity_count": len(nodes),
            "edge_count": len(sub_edges),
            "sar_count": sar_in_case,
            "typologies": ", ".join(typologies) if typologies else "None",
            "seed_entities": ", ".join(sorted(seeds)[:3]) + ("..." if len(seeds) > 3 else ""),
            "seed_count": len(seeds),
        })
    
    cases_df = pd.DataFrame(case_records).sort_values("risk_score", ascending=False).reset_index(drop=True)
    
    # Summary
    typology_counts = defaultdict(int)
    for rec in case_records:
        for t in rec["typologies"].split(", "):
            if t and t != "None":
                typology_counts[t] += 1
    
    summary = {
        "total_cases": len(cases_df),
        "seed_count": len(seed_ids),
        "typology_counts": dict(typology_counts),
        "tier_counts": cases_df["tier"].value_counts().to_dict() if len(cases_df) > 0 else {},
        "total_entities": int(cases_df["entity_count"].sum()) if len(cases_df) > 0 else 0,
        "total_sar": int(cases_df["sar_count"].sum()) if len(cases_df) > 0 else 0,
    }
    
    return cases_df, summary

cases_df, case_summary = build_cases(risk_queue, edges)
print(f"Cases built: {len(cases_df)}")
print(f"Typology counts: {case_summary.get('typology_counts', {})}")

Cases built: 1
Typology counts: {'CIRCULAR_FLOW': 1, 'FAN_IN': 1, 'FAN_OUT': 1, 'HUB_DOMINANCE': 1, 'SAR_PROXIMITY': 1}


---
## 4. Build Dashboard

In [6]:
app = dash.Dash(__name__, external_stylesheets=[dbc.themes.DARKLY])

CLR = {
    "critical": "#DC2626", "high": "#F59E0B", "medium": "#3B82F6", "low": "#22C55E",
    "blue": "#3498db", "green": "#2ecc71", "red": "#e74c3c", "purple": "#9b59b6",
    "orange": "#e67e22", "dark": "#2c3e50"
}
TIER_COLORS = {"T1": CLR["critical"], "T2": CLR["high"], "T3": CLR["medium"], "T4": CLR["low"]}
BAND_COLORS = {"Critical": CLR["critical"], "High": CLR["high"], "Medium": CLR["medium"], "Low": CLR["low"]}

def kpi_card(title, value, color):
    return dbc.Card([
        dbc.CardBody([
            html.H6(title, className="text-muted mb-1", style={"fontSize": "0.85rem"}),
            html.H3(value, className="mb-0", style={"color": color, "fontWeight": "bold"}),
        ])
    ], className="shadow-sm", style={"borderLeft": f"4px solid {color}"})

In [7]:
# ═══════════════════════════════════════════════════════════
# TAB 1: RISK RANKING
# ═══════════════════════════════════════════════════════════

tier_counts = risk_queue["tier"].value_counts()
fig_tier_bar = go.Figure(go.Bar(
    x=["T1 (Critical)", "T2 (High)", "T3 (Medium)", "T4 (Low)"],
    y=[tier_counts.get("T1", 0), tier_counts.get("T2", 0), tier_counts.get("T3", 0), tier_counts.get("T4", 0)],
    marker_color=[TIER_COLORS["T1"], TIER_COLORS["T2"], TIER_COLORS["T3"], TIER_COLORS["T4"]],
    text=[f"{tier_counts.get(t, 0):,}" for t in ["T1", "T2", "T3", "T4"]],
    textposition="outside"
))
fig_tier_bar.update_layout(template="plotly_dark", title="Entities by Tier", yaxis_title="Count", margin=dict(t=40, b=30))

fig_score_dist = go.Figure()
fig_score_dist.add_trace(go.Histogram(x=risk_queue["risk_score"], nbinsx=50, marker_color=CLR["blue"], opacity=0.75))
fig_score_dist.update_layout(template="plotly_dark", title="Risk Score Distribution", xaxis_title="Score", yaxis_title="Count", margin=dict(t=40, b=30))

top20_risk = risk_queue.head(20)[["entity_id", "tier", "risk_score", "degree", "is_sar", "reasons"]].copy()
top20_risk["risk_score"] = top20_risk["risk_score"].round(4)

tab_risk = dbc.Container([
    dbc.Row([
        dbc.Col(kpi_card("Total Entities", f"{len(risk_queue):,}", CLR["blue"]), md=3),
        dbc.Col(kpi_card("T1 (Critical)", f"{tier_counts.get('T1', 0):,}", TIER_COLORS["T1"]), md=3),
        dbc.Col(kpi_card("T2 (High)", f"{tier_counts.get('T2', 0):,}", TIER_COLORS["T2"]), md=3),
        dbc.Col(kpi_card("SAR Entities", f"{risk_queue['is_sar'].sum():,}", CLR["purple"]), md=3),
    ], className="mb-3 g-3"),
    dbc.Row([
        dbc.Col(dcc.Graph(figure=fig_tier_bar), md=6),
        dbc.Col(dcc.Graph(figure=fig_score_dist), md=6),
    ], className="mb-3"),
    dbc.Row([dbc.Col([
        html.H6("Top 20 Highest Risk Entities", className="text-center mb-2"),
        dash_table.DataTable(
            data=top20_risk.to_dict("records"),
            columns=[{"name": c, "id": c} for c in top20_risk.columns],
            sort_action="native", page_size=10,
            style_header={"backgroundColor": "#2c3e50", "color": "white", "fontWeight": "bold"},
            style_cell={"backgroundColor": "#1a1a2e", "color": "white", "fontSize": "12px", "padding": "6px"},
            style_data_conditional=[
                {"if": {"filter_query": "{tier} = 'T1'"}, "backgroundColor": "rgba(220,38,38,0.2)"},
                {"if": {"filter_query": "{tier} = 'T2'"}, "backgroundColor": "rgba(245,158,11,0.2)"},
            ]
        ),
    ], md=12)]),
], fluid=True)

In [8]:
# ═══════════════════════════════════════════════════════════
# TAB 2: AML SCORES
# ═══════════════════════════════════════════════════════════

band_counts = aml_scores["risk_band"].value_counts()
fig_band_bar = go.Figure(go.Bar(
    x=["Critical", "High", "Medium", "Low"],
    y=[band_counts.get("Critical", 0), band_counts.get("High", 0), band_counts.get("Medium", 0), band_counts.get("Low", 0)],
    marker_color=[BAND_COLORS["Critical"], BAND_COLORS["High"], BAND_COLORS["Medium"], BAND_COLORS["Low"]],
    text=[f"{band_counts.get(b, 0):,}" for b in ["Critical", "High", "Medium", "Low"]],
    textposition="outside"
))
fig_band_bar.update_layout(template="plotly_dark", title="Entities by Risk Band", yaxis_title="Count", margin=dict(t=40, b=30))

fig_aml_hist = go.Figure()
for band, color in BAND_COLORS.items():
    subset = aml_scores[aml_scores["risk_band"] == band]
    fig_aml_hist.add_trace(go.Histogram(x=subset["aml_score"], name=band, marker_color=color, opacity=0.7))
fig_aml_hist.update_layout(template="plotly_dark", title="AML Score Distribution by Band", barmode="stack",
                           xaxis_title="AML Score (0-100)", yaxis_title="Count", margin=dict(t=40, b=30))

# Signal radar for top entity
top_entity = aml_scores.iloc[0]
fig_radar = go.Figure(go.Scatterpolar(
    r=[top_entity["sar_signal"], top_entity["laundering_signal"], top_entity["network_signal"],
       top_entity["geo_signal"], top_entity["volume_signal"]],
    theta=["SAR", "Laundering", "Network", "Geo", "Volume"],
    fill="toself", fillcolor="rgba(52,152,219,0.3)", line=dict(color=CLR["blue"])
))
fig_radar.update_layout(template="plotly_dark", title=f"Signal Profile: {top_entity['entity_id'][:12]}",
                        polar=dict(radialaxis=dict(range=[0, 1])), margin=dict(t=40, b=30))

top20_aml = aml_scores.head(20)[["entity_id", "risk_band", "aml_score", "sar_signal", "network_signal", "volume_signal", "aml_reasons"]].copy()

tab_aml = dbc.Container([
    dbc.Row([
        dbc.Col(kpi_card("Total Scored", f"{len(aml_scores):,}", CLR["blue"]), md=3),
        dbc.Col(kpi_card("Critical", f"{band_counts.get('Critical', 0):,}", BAND_COLORS["Critical"]), md=3),
        dbc.Col(kpi_card("High", f"{band_counts.get('High', 0):,}", BAND_COLORS["High"]), md=3),
        dbc.Col(kpi_card("Mean Score", f"{aml_scores['aml_score'].mean():.1f}", CLR["purple"]), md=3),
    ], className="mb-3 g-3"),
    dbc.Row([
        dbc.Col(dcc.Graph(figure=fig_band_bar), md=4),
        dbc.Col(dcc.Graph(figure=fig_aml_hist), md=4),
        dbc.Col(dcc.Graph(figure=fig_radar), md=4),
    ], className="mb-3"),
    dbc.Row([dbc.Col([
        html.H6("Top 20 by AML Score", className="text-center mb-2"),
        dash_table.DataTable(
            data=top20_aml.to_dict("records"),
            columns=[{"name": c, "id": c} for c in top20_aml.columns],
            sort_action="native", page_size=10,
            style_header={"backgroundColor": "#2c3e50", "color": "white", "fontWeight": "bold"},
            style_cell={"backgroundColor": "#1a1a2e", "color": "white", "fontSize": "12px", "padding": "6px"},
            style_data_conditional=[
                {"if": {"filter_query": "{risk_band} = 'Critical'"}, "backgroundColor": "rgba(220,38,38,0.2)"},
                {"if": {"filter_query": "{risk_band} = 'High'"}, "backgroundColor": "rgba(245,158,11,0.2)"},
            ]
        ),
    ], md=12)]),
], fluid=True)

In [10]:
# ═══════════════════════════════════════════════════════════
# TAB 3: CASES
# ═══════════════════════════════════════════════════════════

typo_counts = case_summary.get("typology_counts", {})
fig_typo_bar = go.Figure(go.Bar(
    x=list(typo_counts.keys()) if typo_counts else ["None"],
    y=list(typo_counts.values()) if typo_counts else [0],
    marker_color=[CLR["red"], CLR["orange"], CLR["purple"], CLR["blue"], CLR["green"]][:len(typo_counts)] if typo_counts else [CLR["blue"]],
    text=[f"{v}" for v in typo_counts.values()] if typo_counts else ["0"],
    textposition="outside"
))
fig_typo_bar.update_layout(template="plotly_dark", title="Cases by Typology", yaxis_title="Count", margin=dict(t=40, b=30))

case_tier_counts = cases_df["tier"].value_counts() if len(cases_df) > 0 else pd.Series()
fig_case_tier = go.Figure(go.Pie(
    labels=[f"{t} ({case_tier_counts.get(t, 0)})" for t in ["T1", "T2", "T3", "T4"]],
    values=[case_tier_counts.get(t, 0) for t in ["T1", "T2", "T3", "T4"]],
    marker_colors=[TIER_COLORS["T1"], TIER_COLORS["T2"], TIER_COLORS["T3"], TIER_COLORS["T4"]],
    hole=0.4
))
fig_case_tier.update_layout(template="plotly_dark", title="Cases by Tier", margin=dict(t=40, b=10))

cases_display = cases_df[["case_id", "tier", "risk_score", "entity_count", "edge_count", "sar_count", "typologies"]].copy() if len(cases_df) > 0 else pd.DataFrame()

tab_cases = dbc.Container([
    dbc.Row([
        dbc.Col(kpi_card("Total Cases", f"{case_summary.get('total_cases', 0):,}", CLR["blue"]), md=3),
        dbc.Col(kpi_card("Seed Entities", f"{case_summary.get('seed_count', 0):,}", CLR["purple"]), md=3),
        dbc.Col(kpi_card("Entities in Cases", f"{case_summary.get('total_entities', 0):,}", CLR["green"]), md=3),
        dbc.Col(kpi_card("SARs in Cases", f"{case_summary.get('total_sar', 0):,}", CLR["red"]), md=3),
    ], className="mb-3 g-3"),
    dbc.Row([
        dbc.Col(dcc.Graph(figure=fig_typo_bar), md=6),
        dbc.Col(dcc.Graph(figure=fig_case_tier), md=6),
    ], className="mb-3"),
    dbc.Row([dbc.Col([
        html.H6("Case Index", className="text-center mb-2"),
        dash_table.DataTable(
            data=cases_display.to_dict("records") if len(cases_display) > 0 else [],
            columns=[{"name": c, "id": c} for c in cases_display.columns] if len(cases_display) > 0 else [],
            sort_action="native", page_size=15,
            style_header={"backgroundColor": "#2c3e50", "color": "white", "fontWeight": "bold"},
            style_cell={"backgroundColor": "#1a1a2e", "color": "white", "fontSize": "12px", "padding": "6px"},
            style_data_conditional=[
                {"if": {"filter_query": "{tier} = 'T1'"}, "backgroundColor": "rgba(220,38,38,0.2)"},
                {"if": {"filter_query": "{tier} = 'T2'"}, "backgroundColor": "rgba(245,158,11,0.2)"},
            ]
        ) if len(cases_display) > 0 else html.P("No cases generated (check seed selection criteria)", className="text-muted text-center"),
    ], md=12)]),
], fluid=True)

In [11]:
# ═══════════════════════════════════════════════════════════
# APP LAYOUT
# ═══════════════════════════════════════════════════════════

app.layout = dbc.Container([
    dbc.Row([dbc.Col([
        html.H2("AML Advanced Analytics Dashboard", className="text-center mt-3 mb-1"),
        html.Div([
            dbc.Badge("PyTorch + GraphSAGE", color="info", className="me-2"),
            dbc.Badge(f"{len(node_features):,} nodes", color="secondary", className="me-2"),
            dbc.Badge(f"{len(transactions):,} txns", color="secondary", className="me-2"),
            dbc.Badge(f"{len(cases_df)} cases", color="warning", className="me-2"),
        ], className="text-center mb-3"),
    ], md=12)]),
    dbc.Tabs([
        dbc.Tab(tab_risk, label="Risk Ranking", tab_id="tab-risk"),
        dbc.Tab(tab_aml, label="AML Scores", tab_id="tab-aml"),
        dbc.Tab(tab_cases, label="Cases", tab_id="tab-cases"),
    ], active_tab="tab-risk"),
], fluid=True)

print("Dashboard ready. Run next cell to launch.")

Dashboard ready. Run next cell to launch.


In [12]:
# Launch dashboard at http://localhost:8051
app.run(jupyter_mode="inline", port=8051)

---
## 5. Save Outputs

In [ ]:
# Save risk queue
risk_queue.to_parquet(os.path.join(QUEUES_DIR, "risk_queue.parquet"), index=False)
risk_queue.to_csv(os.path.join(QUEUES_DIR, "risk_queue.csv"), index=False)

# Save AML scores (renamed to match API expectations)
aml_scores.to_parquet(os.path.join(QUEUES_DIR, "party_aml_scores.parquet"), index=False)
aml_scores.to_csv(os.path.join(QUEUES_DIR, "party_aml_scores.csv"), index=False)

# Save cases
if len(cases_df) > 0:
    cases_df.to_parquet(os.path.join(CASES_DIR, "case_index.parquet"), index=False)
    cases_df.to_csv(os.path.join(CASES_DIR, "case_index.csv"), index=False)

# Save summaries
queue_summary = {
    "total_entities": len(risk_queue),
    "tier_counts": risk_queue["tier"].value_counts().to_dict(),
    "sar_count": int(risk_queue["is_sar"].sum()),
    "score_range": {
        "min": float(risk_queue["risk_score"].min()),
        "max": float(risk_queue["risk_score"].max()),
        "mean": float(risk_queue["risk_score"].mean()),
    }
}
with open(os.path.join(QUEUES_DIR, "queue_summary.json"), "w") as f:
    json.dump(queue_summary, f, indent=2)

# Renamed to match API expectations
aml_summary = {
    "total_scored": len(aml_scores),
    "band_distribution": aml_scores["risk_band"].value_counts().to_dict(),
    "mean_score": float(aml_scores["aml_score"].mean()),
    "weights_used": WEIGHTS,
}
with open(os.path.join(QUEUES_DIR, "aml_score_summary.json"), "w") as f:
    json.dump(aml_summary, f, indent=2)

with open(os.path.join(CASES_DIR, "case_summary.json"), "w") as f:
    json.dump(case_summary, f, indent=2)

print("Outputs saved:")
print(f"  {QUEUES_DIR}/risk_queue.parquet")
print(f"  {QUEUES_DIR}/party_aml_scores.parquet")
print(f"  {QUEUES_DIR}/queue_summary.json")
print(f"  {QUEUES_DIR}/aml_score_summary.json")
print(f"  {CASES_DIR}/case_index.parquet")
print(f"  {CASES_DIR}/case_summary.json")
print(f"\n  risk_queue: {len(risk_queue):,} entities")
print(f"  aml_scores: {len(aml_scores):,} entities")
print(f"  cases: {len(cases_df)} cases")

In [ ]:
# ── GPU Cleanup — free VRAM for next notebook ──
import gc
for v in ["G_model", "E_model"]:
    if v in dir():
        exec(f"del {v}")
torch.cuda.empty_cache(); gc.collect()
print(f"GPU freed: {torch.cuda.memory_allocated()/1e6:.1f} MB allocated")

GPU freed: 8.5 MB allocated


: 